# Results for model: microsoft/phi-3.5-mini-instruct

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import numpy as np

# Load the training data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Feature Engineering: Calculate TOP_QUANTILE
def top_quantile_feature(df, feature, quantile):
    df_grouped = df.groupby('responder_6').agg([(f'{feature}_top_{quantile}', pl.quantile(feature, quantile))] + 
                                                [(f'{feature}_count', pl.count())])
    df_expanded = df_grouped.join(df, on='responder_6')
    df_expanded = df_expanded.fillna(0).rename(columns={f'{feature}_top_{quantile}': f'{feature}_top_{quantile}' + '_rank',
                                                         f'{feature}_count': 'count'})
    df_final = df_expanded.sort_values(by['date_id']).groupby('date_id').agg([('mean', pl.mean)] +
                                                                         [('count', 'sum')])
    return df_final[f'{feature}_top_{quantile}_rank']

# Calculate the global TOP_QUANTILE for each feature
for feature in df.columns:
    if feature not in ['date_id', 'responder_6', 'target']:
        df[f'{feature}_top_15%'] = top_quantile_feature(df, feature, 0.15)

# Prepare the dataset
X = df[df.columns.difference(['date_id', 'responder_6', 'target'])].to_numpy()
y = df['target'].to_numpy()

# Split the dataset
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Train XGBoost Regressor
model = xgb.XGBRegressor(objective='reg:squarederror')
model.fit(X_train, y_train)

# Make predictions
predictions = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, predictions)
print(f'Mean Squared Error: {mse}')